# V33 — multi-event, event-centred Demucs fine-tuning and Phase-2 submission

This restart-safe notebook reuses the completed Indic Parler-TTS waveforms and
BEATs-matched validation residual bank. It does **not** rerun TTS, WavLM or
BEATs. It creates reproducible 1–3-event mixtures, initializes from the proven
V32 Demucs epoch 6 checkpoint, fine-tunes for three low-learning-rate epochs,
evaluates V32 and every V33 checkpoint on the same real development sets, and
packages an always-on Phase-2 submission.

Training-time residual matching never appears in inference. Every submitted
clip is processed by the same Demucs model without filename, threshold,
natural/synthetic, classifier or dictionary routing.


## 0 — One-cell startup preflight (run this first)


In [1]:
# Run first in a fresh Colab GPU runtime. No TTS/BEATs/WavLM imports are needed.
import os, sys, subprocess, importlib.util, importlib.metadata
from pathlib import Path

os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").is_dir():
        drive.mount("/content/drive")
except ImportError as exc:
    raise RuntimeError("This notebook is designed for Google Colab.") from exc

required = {
    "numpy":"numpy", "pandas":"pandas", "soundfile":"soundfile", "librosa":"librosa",
    "scipy":"scipy", "tqdm":"tqdm", "huggingface_hub":"huggingface_hub",
}
missing = [pkg for module,pkg in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable,"-m","pip","install","-q",*missing])

try: transformers_version=importlib.metadata.version("transformers")
except importlib.metadata.PackageNotFoundError: transformers_version=None
try: protobuf_version=importlib.metadata.version("protobuf")
except importlib.metadata.PackageNotFoundError: protobuf_version=None
need_stack=(transformers_version!="4.46.1" or protobuf_version!="6.31.1" or
            importlib.util.find_spec("denoiser") is None)
if need_stack:
    if "transformers" in sys.modules or "google.protobuf" in sys.modules:
        raise RuntimeError("Restart once: transformers/protobuf was imported before this startup cell.")
    subprocess.check_call([sys.executable,"-m","pip","install","-q","--upgrade",
                           "transformers==4.46.1","tokenizers==0.20.3","protobuf==6.31.1"])
    if importlib.util.find_spec("denoiser") is None:
        subprocess.check_call([sys.executable,"-m","pip","install","-q","--no-deps","denoiser==0.1.5"])

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab GPU runtime before continuing.")
from google.protobuf import runtime_version as _protobuf_check
from transformers import AutoModel as _transformers_check
from denoiser import pretrained as _demucs_check
_m=_demucs_check.dns64(pretrained=False)
if _m.sample_rate!=16000 or _m.chin!=1:
    raise RuntimeError("Unexpected Demucs configuration")
del _m,_protobuf_check,_transformers_check,_demucs_check

root=Path("/content/drive/MyDrive/PHD/Datathon")
v32=root/"track2_phase2_v32_indicparler_beats_demucs"/"run_v1"
prep_candidates=[root/"track2_phase2_v31_dhvaani_beats"/"run_v1",v32]
prep=next((p for p in prep_candidates if (p/"noise_profiles_train.csv").is_file() and
           (p/"source_profile_matches").is_dir()),None)
pin_candidates=[
    Path("/content/drive/MyDrive/Datathon/track2_v6_direct_reconstruction/sravaani_revision_pin.json"),
    root/"track2_v6_direct_reconstruction"/"sravaani_revision_pin.json",
]
required_paths=[
    root/"validation"/"validationMetadata.json", root/"input_data.zip", root/"transcripts.jsonl",
    v32/"parler_clean_16k", v32/"demucs_epoch_06.pt",
]
if prep is None: required_paths.append(root/"track2_phase2_v31_dhvaani_beats"/"run_v1"/"noise_profiles_train.csv")
if not any(p.is_file() for p in pin_candidates): required_paths.append(pin_candidates[-1])
missing_paths=[str(p) for p in required_paths if not p.exists()]
if missing_paths:
    raise FileNotFoundError("Startup preflight missing:\n"+"\n".join(missing_paths))
print("STARTUP PREFLIGHT PASS | CUDA",torch.cuda.get_device_name(0),
      "| transformers",importlib.metadata.version("transformers"),
      "| protobuf",importlib.metadata.version("protobuf"),
      "| denoiser",importlib.metadata.version("denoiser"),"| preparation cache",prep)


Mounted at /content/drive
STARTUP PREFLIGHT PASS | CUDA Tesla T4 | transformers 4.46.1 | protobuf 6.31.1 | denoiser 0.1.5 | preparation cache /content/drive/MyDrive/PHD/Datathon/track2_phase2_v31_dhvaani_beats/run_v1


## 1 — Configuration and exact audio/submission contract


In [2]:
from pathlib import Path
import os, io, re, gc, json, math, wave, shutil, random, hashlib, zipfile, unicodedata, time
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
from scipy.ndimage import median_filter
from tqdm.auto import tqdm
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from huggingface_hub import hf_hub_download, snapshot_download
from IPython.display import display, Audio

ROOT = Path("/content/drive/MyDrive/PHD/Datathon")
VAL_ROOT = ROOT / "validation"
META_PATH = VAL_ROOT / "validationMetadata.json"
SOURCE_AUDIO = ROOT / "input_data.zip"
TRANSCRIPTS_JSONL = ROOT / "transcripts.jsonl"

V32_RUN = ROOT / "track2_phase2_v32_indicparler_beats_demucs" / "run_v1"
OUT = ROOT / "track2_phase2_v33_multievent_demucs"
LOCAL = Path("/content/track2_phase2_v33_multievent_demucs")
RUN = OUT / "run_v1"
WORK = LOCAL / "run_v1"
for p in (OUT, LOCAL, RUN, WORK): p.mkdir(parents=True, exist_ok=True)
V31_PREP_CACHE = ROOT / "track2_phase2_v31_dhvaani_beats" / "run_v1"
PREP_CACHE = V31_PREP_CACHE if (V31_PREP_CACHE/"noise_profiles_train.csv").is_file() else V32_RUN
TTS_DIR = V32_RUN / "parler_clean_16k"
MATCH_DIR = PREP_CACHE / "source_profile_matches"
PROFILE_MANIFEST = PREP_CACHE / "noise_profiles_train.csv"
BASELINE_CHECKPOINT = V32_RUN / "demucs_epoch_06.pt"

SR = 16000
SEED = 517
PSEUDO_PAIRS = 5000
EVENT_COUNT_VALUES = (1,2,3)
EVENT_COUNT_PROBABILITIES = (.40,.40,.20)
SNR_JITTER_DB = 4.0
EVENT_CENTRED_PROBABILITY = .70
RANDOM_CROP_PROBABILITY = .15
IDENTITY_PROBABILITY = .15
NATURAL_DEV_LIMIT = 100
NATURAL_MONITOR_LIMIT = 100
SEGMENT = 51200                 # 3.2 s
BATCH_SIZE = 4
FINETUNE_EPOCHS = 3
LR_FULL = 2e-6
CHUNK_SAMPLES = 51200
CHUNK_HOP = 25600
EXPECTED_PHASE2_CLIPS = 3178
RUN_PHASE2 = True

# The user has confirmed that the competition rules permit this source-derived
# training experiment. This does not permit any source-specific inference route.
CONFIRM_RULES_ALLOW_SOURCE_DERIVED_TRAINING = True

ASR_REPO = "ARTPARK-IISc/SraVaani-1.0"
ASR_PIN_CANDIDATES = [
    Path("/content/drive/MyDrive/Datathon/track2_v6_direct_reconstruction/sravaani_revision_pin.json"),
    ROOT / "track2_v6_direct_reconstruction" / "sravaani_revision_pin.json",
]
ASR_PIN = next((p for p in ASR_PIN_CANDIDATES if p.is_file()), ASR_PIN_CANDIDATES[-1])
DEVICE = torch.device("cuda")
IMPLEMENTATION = "v33_multievent_eventcentred_demucs_v1"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print("Persistent V33 run:",RUN)
print("V32 baseline:",BASELINE_CHECKPOINT)
print("Reusable TTS:",TTS_DIR)
print("Reusable profile/match cache:",PREP_CACHE)


Persistent V33 run: /content/drive/MyDrive/PHD/Datathon/track2_phase2_v33_multievent_demucs/run_v1
V32 baseline: /content/drive/MyDrive/PHD/Datathon/track2_phase2_v32_indicparler_beats_demucs/run_v1/demucs_epoch_06.pt
Reusable TTS: /content/drive/MyDrive/PHD/Datathon/track2_phase2_v32_indicparler_beats_demucs/run_v1/parler_clean_16k
Reusable profile/match cache: /content/drive/MyDrive/PHD/Datathon/track2_phase2_v31_dhvaani_beats/run_v1


In [3]:
def jsonable(x):
    if isinstance(x, dict): return {str(k):jsonable(v) for k,v in x.items()}
    if isinstance(x, (list, tuple)): return [jsonable(v) for v in x]
    if isinstance(x, Path): return str(x)
    if isinstance(x, np.ndarray): return jsonable(x.tolist())
    if isinstance(x, np.generic): return jsonable(x.item())
    if isinstance(x, float) and not math.isfinite(x): return None
    return x

def put_json(path, obj):
    p=Path(path); p.parent.mkdir(parents=True,exist_ok=True)
    t=p.with_name(p.name+'.tmp')
    with open(t,'w',encoding='utf-8') as f:
        json.dump(jsonable(obj),f,indent=2,ensure_ascii=False,allow_nan=False)
    os.replace(t,p)

def sha_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(4<<20),b''): h.update(b)
    return h.hexdigest()

def sha_json(obj):
    return hashlib.sha256(json.dumps(jsonable(obj),sort_keys=True,separators=(',',':'),allow_nan=False).encode()).hexdigest()

def canonical_pcm(y):
    a=np.asarray(y,np.float64)
    if a.ndim!=1 or not len(a) or not np.isfinite(a).all():
        raise ValueError('Wave must be finite, nonempty, and mono.')
    # Deterministic rounding, not peak normalization. Used for metrics AND export.
    return np.clip(np.rint(a*32768.0),-32768,32767).astype('<i2')

def pcm_roundtrip(y): return canonical_pcm(y).astype(np.float64)/32768.0

def wav_bytes(y, sr=16000):
    buf=io.BytesIO()
    with wave.open(buf,'wb') as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(sr)
        w.writeframes(canonical_pcm(y).tobytes())
    return buf.getvalue()

def write_pcm(path,y,sr=16000):
    p=Path(path); p.parent.mkdir(parents=True,exist_ok=True)
    data=wav_bytes(y,sr); tmp=p.with_name(p.name+'.tmp')
    with open(tmp,'wb') as f: f.write(data)
    os.replace(tmp,p)
    return hashlib.sha256(data).hexdigest()

def read_wave_strict(path_or_bytes,sr=16000):
    source=io.BytesIO(path_or_bytes) if isinstance(path_or_bytes,(bytes,bytearray)) else path_or_bytes
    y,fs=sf.read(source,dtype='float32',always_2d=True)
    if fs!=sr or y.shape[1]!=1:
        raise ValueError(f'Expected native {sr}-Hz mono audio, got {fs} Hz / {y.shape[1]} channels. No unvalidated resampling.')
    a=y[:,0]
    if not len(a) or not np.isfinite(a).all(): raise ValueError('Empty or invalid audio.')
    return a

def wav_audit(path_or_bytes,expected_frames,sr=16000):
    data=path_or_bytes if isinstance(path_or_bytes,bytes) else Path(path_or_bytes).read_bytes()
    with wave.open(io.BytesIO(data),'rb') as w:
        got=(w.getframerate(),w.getnchannels(),w.getsampwidth(),w.getnframes(),w.getcomptype())
        wanted=(sr,1,2,int(expected_frames),'NONE')
        if got!=wanted: raise ValueError(f'WAV compliance error: {got} != {wanted}')
        pcm=w.readframes(expected_frames)
        if len(pcm)!=expected_frames*2: raise ValueError('Truncated PCM payload.')
    return hashlib.sha256(data).hexdigest()

class StrictWaveSource:
    """One explicit directory/ZIP, exact IDs; no duplicate stems or inferred paths."""
    def __init__(self,path,required_ids):
        self.path=str(path); self.archive=None; self.index={}
        required_ids=set(map(str,required_ids))
        p=Path(path)
        if p.is_file() and zipfile.is_zipfile(p):
            self.archive=zipfile.ZipFile(p)
            names=[z.filename for z in self.archive.infolist() if not z.is_dir() and z.filename.lower().endswith('.wav') and '__MACOSX' not in z.filename]
        elif p.is_dir(): names=[str(q) for q in p.rglob('*') if q.is_file() and q.suffix.lower()=='.wav']
        else: raise FileNotFoundError(p)
        for name in names:
            cid=Path(name).stem
            if cid not in required_ids: continue
            if cid in self.index: raise ValueError(f'Duplicate waveform ID: {cid} in {p}')
            self.index[cid]=name
        missing=required_ids-set(self.index)
        if missing: raise ValueError(f'{p}: {len(missing)} missing IDs; examples {sorted(missing)[:3]}')
    def bytes(self,cid):
        loc=self.index[cid]
        return self.archive.read(loc) if self.archive is not None else Path(loc).read_bytes()
    def load(self,cid): return read_wave_strict(self.bytes(cid))
    def close(self):
        if self.archive is not None: self.archive.close()

def si_sdr(estimate,reference):
    """Zero-mean SI-SDR, no alignment/resampling; 120-dB numerical ceiling only."""
    x=np.asarray(estimate,np.float64); s=np.asarray(reference,np.float64)
    if x.shape!=s.shape or x.ndim!=1: raise ValueError('SI-SDR length/shape mismatch.')
    if not np.isfinite(x).all() or not np.isfinite(s).all(): raise ValueError('Nonfinite SI-SDR input.')
    x=x-x.mean(); s=s-s.mean(); ss=float(s@s)
    if ss<=1e-18: raise ValueError('Silent reference: SI-SDR undefined, not silently excluded.')
    target=(float(x@s)/ss)*s; e=x-target
    num=float(target@target); den=float(e@e)
    value=10*math.log10(max(num,ss*1e-16)/max(den,ss*1e-12))
    return float(min(120.,value))

def hyp_text(h):
    if isinstance(h,str): return h.strip()
    if isinstance(h,dict) and isinstance(h.get('text'),str): return h['text'].strip()
    if hasattr(h,'text') and isinstance(h.text,str): return h.text.strip()
    if isinstance(h,(list,tuple)) and len(h)==1: return hyp_text(h[0])
    raise TypeError(f'Unrecognized ASR hypothesis type: {type(h)}. Refusing an invented empty transcript.')

def pack_submission(paths,asr,expected_frames,output_zip,sr=16000):
    """Only final audio and matching ASR enter the ZIP; diagnostics/clean refs never do."""
    ids=sorted(paths)
    if set(ids)!=set(asr) or set(ids)!=set(expected_frames): raise ValueError('Submission ID mismatch.')
    for cid in ids:
        if not cid or Path(cid).name!=cid or any(c in cid for c in ('/','\\','\n','\r')): raise ValueError('Unsafe clip ID.')
        h=wav_audit(paths[cid],expected_frames[cid],sr)
        if h!=asr[cid]['wav_sha256']: raise ValueError(f'ASR/audio hash mismatch: {cid}')
        if not isinstance(asr[cid]['text'],str): raise ValueError('ASR text must be a string.')
    p=Path(output_zip); p.parent.mkdir(parents=True,exist_ok=True); tmp=p.with_name(p.name+'.tmp')
    tr=''.join(json.dumps(dict(clip_id=c,text=asr[c]['text']),ensure_ascii=False,allow_nan=False)+'\n' for c in ids).encode('utf-8')
    with zipfile.ZipFile(tmp,'w',zipfile.ZIP_DEFLATED,compresslevel=9,allowZip64=True) as z:
        z.writestr('transcripts.jsonl',tr)
        for cid in ids: z.write(paths[cid],cid+'.wav')
    with zipfile.ZipFile(tmp) as z:
        wanted={'transcripts.jsonl'}|{c+'.wav' for c in ids}
        if len(z.namelist())!=len(wanted) or set(z.namelist())!=wanted: raise ValueError('Archive contents mismatch.')
        if z.testzip() is not None: raise ValueError('Archive CRC failure.')
        for cid in ids:
            data=z.read(cid+'.wav')
            if wav_audit(data,expected_frames[cid],sr)!=asr[cid]['wav_sha256']: raise ValueError('Packaged audio mismatch.')
        rows=[json.loads(x) for x in z.read('transcripts.jsonl').decode().splitlines()]
        if len(rows)!=len(ids) or {r['clip_id'] for r in rows}!=set(ids): raise ValueError('Packaged transcript mismatch.')
    os.replace(tmp,p)
    return dict(zip_path=str(p),zip_sha256=sha_file(p),clips=len(ids),transcripts_sha256=hashlib.sha256(tr).hexdigest())


## 2 — Reconstruct the fixed real validation splits


In [4]:
def wav_index(root):
    if not root.is_dir(): raise FileNotFoundError(root)
    result={}
    for p in sorted(root.rglob('*')):
        if p.is_file() and p.suffix.lower()=='.wav':
            if p.name in result: raise ValueError(f'Duplicate filename in {root}: {p.name}')
            result[p.name]=p
    return result

def filename_group(name):
    # Dataset filenames end the apparent participant/session token with digits,
    # followed by a numeric utterance token. Retain prefix before that token.
    m=re.match(r'^(.*?\d{6,})_\d+_',Path(name).stem)
    return m.group(1) if m else Path(name).stem

def make_validation_tables(meta_path,root):
    records=json.loads(meta_path.read_text(encoding='utf-8'))
    if not isinstance(records,list): raise ValueError('Expected a JSON list')
    names=[r['segmentFileName'] for r in records]
    if len(names)!=len(set(names)): raise ValueError('Duplicate metadata filenames')
    noisy=wav_index(root/'syntheticNoiseAudio')
    clean=wav_index(root/'syntheticCleanRefAudio')
    natural=wav_index(root/'naturalNoisyAudio')
    pairs=[]; nat=[]
    for r in tqdm(records,desc='Audit validation audio'):
        name=r['segmentFileName']; cid=Path(name).stem
        if Path(name).name!=name: raise ValueError('Expected basename in segmentFileName')
        if not isinstance(r['syntheticData'],bool): raise ValueError('syntheticData must be Boolean')
        common=dict(clip_id=cid,transcript=r.get('transcript',''),language=r.get('language',''),
                    events=r.get('NoiseSubCategoryTimeStamp',[]))
        if r['syntheticData']:
            if name not in noisy or name not in clean: raise FileNotFoundError(f'Missing pair: {name}')
            x=read_wave_strict(noisy[name]); s=read_wave_strict(clean[name])
            if x.shape!=s.shape: raise ValueError(f'Pair length mismatch: {name}')
            if np.array_equal(x,s): raise ValueError(f'Identical noisy/clean pair: {name}')
            if np.mean(s.astype(np.float64)**2)<1e-10: raise ValueError(f'Silent clean target: {name}')
            pairs.append(dict(**common,noisy_path=str(noisy[name]),clean_path=str(clean[name]),
                group_proxy=filename_group(name),frames=len(x),
                noisy_hash=hashlib.sha256(x.tobytes()).hexdigest(),
                clean_hash=hashlib.sha256(s.tobytes()).hexdigest()))
        else:
            if name not in natural: raise FileNotFoundError(f'Missing natural WAV: {name}')
            x=read_wave_strict(natural[name])
            nat.append(dict(**common,noisy_path=str(natural[name]),frames=len(x),
                            noisy_hash=hashlib.sha256(x.tobytes()).hexdigest()))
    # Union grouping proxies and exact clean/noisy duplicates, including cross roles.
    parent=list(range(len(pairs)))
    def find(i):
        while parent[i]!=i:
            parent[i]=parent[parent[i]]; i=parent[i]
        return i
    def union(a,b): parent[find(a)]=find(b)
    seen={}
    for i,r in enumerate(pairs):
        for key in [('proxy',r['group_proxy']),('audio',r['clean_hash']),('audio',r['noisy_hash'])]:
            if key in seen: union(i,seen[key])
            else: seen[key]=i
    groups={}
    for i in range(len(pairs)): groups.setdefault(find(i),[]).append(i)
    if len(groups)<10: raise ValueError('Too few independent groups for a useful split')
    ordered=sorted(groups.values(),key=lambda ids: hashlib.sha256(
        (str(SEED)+'|'+min(pairs[i]['clip_id'] for i in ids)).encode()).hexdigest())
    ng=len(ordered); n_hold=max(1,round(.15*ng)); n_dev=max(1,round(.15*ng))
    for j,ids in enumerate(ordered):
        split='holdout' if j<n_hold else 'dev' if j<n_hold+n_dev else 'train'
        group=min(pairs[i]['clip_id'] for i in ids)
        for i in ids: pairs[i].update(split=split,split_group=group)
    return pd.DataFrame(pairs),pd.DataFrame(nat)

PAIRS,NATURAL=make_validation_tables(META_PATH,VAL_ROOT)
if len(PAIRS)!=695 or len(NATURAL)!=1806:
    raise ValueError(f'Metadata changed: {len(PAIRS)} paired / {len(NATURAL)} natural. Inspect before proceeding.')
DATA_SHA=sha_json(dict(metadata=sha_file(META_PATH),audio=PAIRS[
    ['clip_id','noisy_hash','clean_hash','split']].to_dict('records')))

def natural_sample(df,exclude_ids=(),limit=None):
    rows=df[~df.clip_id.isin(exclude_ids)].copy()
    rows['_key']=rows.clip_id.map(lambda s:hashlib.sha256((str(SEED)+s).encode()).hexdigest())
    rows=rows.sort_values('_key')
    return rows if limit is None else rows.head(limit)
# Prior V27 natural sample is explicitly the development set, not a new holdout.
NAT_DEV=natural_sample(NATURAL,limit=NATURAL_DEV_LIMIT)
NAT_MON=natural_sample(NATURAL,exclude_ids=set(NAT_DEV.clip_id),limit=NATURAL_MONITOR_LIMIT)
SYN_DEV=PAIRS[PAIRS.split=='dev'].copy()
SYN_MON=PAIRS[PAIRS.split=='holdout'].copy()
TRAIN_ROWS=PAIRS[PAIRS.split=='train'].copy()
assert set(NAT_DEV.clip_id).isdisjoint(NAT_MON.clip_id)
print('Pairs by split:',PAIRS.groupby('split').size().to_dict())
print('Natural development / additional monitoring:',len(NAT_DEV),len(NAT_MON))


Audit validation audio:   0%|          | 0/2501 [00:00<?, ?it/s]

Pairs by split: {'dev': 100, 'holdout': 104, 'train': 491}
Natural development / additional monitoring: 100 100


## 3 — Rebuild pseudo pairs as 1–3 matched noise events (no TTS/BEATs rerun)


In [ ]:
def event_dicts(raw):
    if raw is None:return []
    if isinstance(raw,str):
        raw=raw.strip()
        if not raw:return []
        raw=json.loads(raw)
    if isinstance(raw,np.ndarray):raw=raw.tolist()
    out=[]
    for e in raw if isinstance(raw,(list,tuple)) else []:
        if not isinstance(e,dict):continue
        start=e.get("start",e.get("onset"));end=e.get("end",e.get("offset"))
        if start is None or end is None:continue
        item=dict(start=float(start),end=float(end),category=str(e.get("category","unknown")),
                  tag=str(e.get("tag","unknown")))
        for key in ("profile_id","match_score","snr_db","noise_attenuation_db"):
            if e.get(key) is not None:item[key]=e[key]
        out.append(item)
    return out

def clean_text(text):
    text=re.sub(r"<[^>]*>|\[[^\]]*\]"," ",str(text or ""))
    return re.sub(r"\s+"," ",unicodedata.normalize("NFC",text)).strip()

TRANSCRIPTS=[json.loads(line) for line in TRANSCRIPTS_JSONL.read_text(encoding="utf-8").splitlines() if line.strip()]
TRANSCRIPT_MAP={str(r["clip_id"]):clean_text(r["text"]) for r in TRANSCRIPTS}
if len(TRANSCRIPT_MAP)!=EXPECTED_PHASE2_CLIPS:raise ValueError("Unexpected transcript count")

PROFILES=pd.read_csv(PROFILE_MANIFEST)
required_profile_columns={"profile_id","path","category","snr_db","split"}
if required_profile_columns-set(PROFILES.columns):
    raise ValueError(f"Profile manifest misses {sorted(required_profile_columns-set(PROFILES.columns))}")
if set(PROFILES.split)!={"train"}:raise RuntimeError("Only training-split validation residuals may be used")
profile_lookup={str(r["profile_id"]):r for r in PROFILES.to_dict("records")}
missing_profiles=[pid for pid,r in profile_lookup.items() if not Path(r["path"]).is_file()]
if missing_profiles:raise FileNotFoundError(f"Missing {len(missing_profiles)} residual WAVs; examples {missing_profiles[:3]}")

match_map={p.stem:json.loads(p.read_text()) for p in MATCH_DIR.glob("*.json")}
eligible=sorted(cid for cid in TRANSCRIPT_MAP if (TTS_DIR/(cid+".wav")).is_file() and
                cid in match_map and match_map[cid].get("events"))
if len(eligible)!=EXPECTED_PHASE2_CLIPS:
    raise RuntimeError(f"Expected all {EXPECTED_PHASE2_CLIPS} cached TTS/match clips, found {len(eligible)}")

PSEUDO_DIR=RUN/"pseudo_pairs_multievent";NOISY_DIR=PSEUDO_DIR/"noisy"
NOISY_DIR.mkdir(parents=True,exist_ok=True)
PSEUDO_MANIFEST=RUN/"pseudo_pairs_multievent.jsonl"
PAIR_CONFIG=dict(version="v33_multievent_v1",seed=SEED,pairs=PSEUDO_PAIRS,
    event_counts=list(EVENT_COUNT_VALUES),event_probabilities=list(EVENT_COUNT_PROBABILITIES),
    snr_jitter_db=SNR_JITTER_DB,tts_dir=str(TTS_DIR),profile_manifest_sha=sha_file(PROFILE_MANIFEST))
PAIR_CONFIG_SHA=sha_json(PAIR_CONFIG)
pair_config_path=RUN/"pseudo_pair_config.json"
if pair_config_path.exists():
    old=json.loads(pair_config_path.read_text())
    if old.get("sha256")!=PAIR_CONFIG_SHA:
        raise RuntimeError("V33 pair configuration changed. Use a new RUN directory rather than mixing caches.")
else:put_json(pair_config_path,dict(config=PAIR_CONFIG,sha256=PAIR_CONFIG_SHA))

rng=np.random.default_rng(SEED+33001)
source_order=list(eligible)
while len(source_order)<PSEUDO_PAIRS:source_order.extend(rng.permutation(eligible).tolist())
source_order=source_order[:PSEUDO_PAIRS]

def candidate_profiles(match):
    candidates=[]
    for event in match.get("events",[]):
        candidates.extend(event.get("profiles",[]))
    best={}
    for item in candidates:
        pid=str(item["profile_id"])
        if pid not in profile_lookup:continue
        if pid not in best or float(item.get("score",-1e9))>float(best[pid].get("score",-1e9)):
            best[pid]=dict(profile_id=pid,score=float(item.get("score",0.0)),
                           category=str(item.get("category",profile_lookup[pid]["category"])))
    return sorted(best.values(),key=lambda z:z["score"],reverse=True)

def choose_profiles(options,count,rng):
    if len(options)<count:raise RuntimeError("Fewer matched profiles than requested events")
    remaining=list(options);chosen=[]
    while len(chosen)<count:
        scores=np.asarray([o["score"] for o in remaining],np.float64)
        weights=np.exp(np.clip((scores-scores.max())/.15,-20,0))
        used={o["category"] for o in chosen}
        weights*=np.asarray([1.6 if o["category"] not in used else 1.0 for o in remaining])
        weights/=weights.sum();j=int(rng.choice(len(remaining),p=weights))
        chosen.append(remaining.pop(j))
    return chosen

def fit_noise(noise,max_len,rng):
    if len(noise)<=max_len:return noise
    start=int(rng.integers(0,len(noise)-max_len+1));return noise[start:start+max_len]

def maximum_safe_mix_gain(clean,noise_mix,limit=.98):
    if np.max(np.abs(clean+noise_mix))<=limit:return 1.0
    lo,hi=0.0,1.0
    for _ in range(32):
        mid=(lo+hi)/2
        if np.max(np.abs(clean+mid*noise_mix))<=limit:lo=mid
        else:hi=mid
    return lo

def make_multievent_pair(clean,options,event_count,rng):
    mix=np.zeros_like(clean);placed=[];previous=None
    for choice in choose_profiles(options,event_count,rng):
        pr=profile_lookup[choice["profile_id"]]
        noise=fit_noise(read_wave_strict(pr["path"]),len(clean),rng)
        if len(noise)<int(.10*SR):raise ValueError("Residual profile is too short")
        max_start=max(0,len(clean)-len(noise))
        if previous is not None and rng.random()<.35:
            low=max(0,previous[0]-len(noise)//2);high=min(max_start,previous[1])
            start=int(rng.integers(low,high+1)) if high>=low else int(rng.integers(0,max_start+1))
        else:start=int(rng.integers(0,max_start+1))
        speech=clean[start:start+len(noise)]
        srms=float(np.sqrt(np.mean(speech.astype(np.float64)**2)+1e-10))
        if srms<1e-3:srms=float(np.sqrt(np.mean(clean.astype(np.float64)**2)+1e-10))
        nrms=float(np.sqrt(np.mean(noise.astype(np.float64)**2)+1e-10))
        snr=float(np.clip(float(pr["snr_db"])+rng.uniform(-SNR_JITTER_DB,SNR_JITTER_DB),-12,18))
        scaled=noise*(srms/(max(nrms,1e-8)*10**(snr/20)))
        mix[start:start+len(scaled)]+=scaled
        previous=(start,start+len(scaled))
        placed.append(dict(start=start/SR,end=(start+len(scaled))/SR,profile_id=choice["profile_id"],
            category=choice["category"],match_score=choice["score"],snr_db=snr))
    alpha=maximum_safe_mix_gain(clean,mix)
    noisy=clean+alpha*mix
    attenuation=20*math.log10(max(alpha,1e-8))
    for e in placed:e["noise_attenuation_db"]=attenuation
    return noisy.astype(np.float32),placed,float(alpha)

existing=[]
if PSEUDO_MANIFEST.exists():
    existing=[json.loads(line) for line in PSEUDO_MANIFEST.read_text(encoding="utf-8").splitlines() if line.strip()]
by_id={r["pair_id"]:r for r in existing}

def save_records(records):
    tmp=PSEUDO_MANIFEST.with_suffix(".tmp")
    tmp.write_text("".join(json.dumps(r,ensure_ascii=False)+"\n" for r in records),encoding="utf-8")
    os.replace(tmp,PSEUDO_MANIFEST)

records=[]
for i,cid in enumerate(tqdm(source_order,desc="Resume 1–3-event pseudo-pair generation")):
    pair_id=f"v33_{i:05d}_{cid}";npth=NOISY_DIR/(pair_id+".wav")
    cached=by_id.get(pair_id)
    if cached is not None and npth.is_file() and cached.get("config_sha")==PAIR_CONFIG_SHA:
        records.append(cached);continue
    local_rng=np.random.default_rng(SEED+i*104729)
    clean=read_wave_strict(TTS_DIR/(cid+".wav"))
    event_count=int(local_rng.choice(EVENT_COUNT_VALUES,p=EVENT_COUNT_PROBABILITIES))
    options=candidate_profiles(match_map[cid])
    noisy,placed,alpha=make_multievent_pair(clean,options,event_count,local_rng)
    if np.array_equal(canonical_pcm(noisy),canonical_pcm(clean)):raise RuntimeError(f"No effective noise in {pair_id}")
    write_pcm(npth,noisy)
    split="pseudo_val" if int(hashlib.sha256(cid.encode()).hexdigest(),16)%10==0 else "pseudo_train"
    records.append(dict(pair_id=pair_id,source_clip=cid,split=split,noisy_path=str(npth),
        clean_path=str(TTS_DIR/(cid+".wav")),frames=len(clean),events=placed,
        event_count=len(placed),noise_mix_gain=alpha,config_sha=PAIR_CONFIG_SHA))
    if (i+1)%100==0:save_records(records)
save_records(records)
PSEUDO=pd.DataFrame(records)

required_columns={"pair_id","source_clip","split","noisy_path","clean_path","frames","events","event_count"}
if required_columns-set(PSEUDO.columns):raise ValueError("Incomplete V33 pseudo manifest")
if len(PSEUDO)!=PSEUDO_PAIRS or PSEUDO.pair_id.duplicated().any():raise ValueError("Wrong/duplicate V33 pair count")
if PSEUDO.source_clip.nunique()!=EXPECTED_PHASE2_CLIPS:raise ValueError("V33 does not cover all source clips")
if not PSEUDO.event_count.between(1,3).all():raise ValueError("Event count outside 1–3")
counts=PSEUDO.event_count.value_counts(normalize=True).sort_index()
profile_ids={str(e["profile_id"]) for events in PSEUDO.events for e in event_dicts(events)}
if float((PSEUDO.event_count>=2).mean())<.55 or float(PSEUDO.event_count.mean())<1.65:
    raise RuntimeError(f"Multi-event generation collapsed: {counts.to_dict()}")
if len(profile_ids)<200:raise RuntimeError(f"Only {len(profile_ids)} residual profiles used")

audit=[]
for r in tqdm(PSEUDO.sample(min(500,len(PSEUDO)),random_state=SEED).to_dict("records"),desc="Audit V33 pairs"):
    x=read_wave_strict(r["noisy_path"]);s=read_wave_strict(r["clean_path"])
    if x.shape!=s.shape or np.array_equal(canonical_pcm(x),canonical_pcm(s)):raise RuntimeError(r["pair_id"])
    audit.append(dict(pair_id=r["pair_id"],duration=len(x)/SR,noisy_rms=float(np.sqrt(np.mean(x.astype(np.float64)**2))),
        clean_rms=float(np.sqrt(np.mean(s.astype(np.float64)**2))),noisy_si_sdr=si_sdr(x,s),events=len(r["events"])))
AUDIT=pd.DataFrame(audit);AUDIT.to_csv(RUN/"pseudo_pair_audit.csv",index=False)
display(PSEUDO.event_count.value_counts().sort_index().rename("pairs").to_frame())
display(AUDIT.describe())
print("V33 PAIR AUDIT PASS | source clips",PSEUDO.source_clip.nunique(),"| profiles",len(profile_ids),
      "| mean events",float(PSEUDO.event_count.mean()),"| multi-event fraction",float((PSEUDO.event_count>=2).mean()))


Resume 1–3-event pseudo-pair generation:   0%|          | 0/5000 [00:00<?, ?it/s]

Audit V33 pairs:   0%|          | 0/500 [00:00<?, ?it/s]

,pairs
event_count,
1,2034
2,1962
3,1004


,duration,noisy_rms,clean_rms,noisy_si_sdr,events
count,500.000000,500.000000,500.000000,500.000000,500.000000
mean,12.987339,0.067079,0.055922,4.824460,1.744000
std,4.936106,0.025423,0.023626,5.268469,0.739972
min,2.414938,0.002853,0.001228,-7.199366,1.000000
25%,9.183500,0.049850,0.039512,1.327907,1.000000
50%,12.416906,0.066955,0.055015,4.275198,2.000000
75%,15.548703,0.084098,0.070574,7.395999,2.000000
max,31.009000,0.151231,0.146859,29.743929,3.000000


V33 PAIR AUDIT PASS | source clips 3178 | profiles 544 | mean events 1.794 | multi-event fraction 0.5932


## 4 — Initialize exactly from the proven V32 Demucs epoch 6


In [ ]:
if importlib.util.find_spec("denoiser") is None:
    subprocess.check_call([sys.executable,"-m","pip","install","-q","--no-deps","denoiser==0.1.5"])
from denoiser import pretrained as demucs_pretrained

model=demucs_pretrained.dns64(pretrained=False)
if model.sample_rate!=SR or model.chin!=1:raise RuntimeError("Demucs configuration is not 16-kHz mono")
if not BASELINE_CHECKPOINT.is_file():raise FileNotFoundError(BASELINE_CHECKPOINT)
baseline_state=torch.load(BASELINE_CHECKPOINT,map_location="cpu",weights_only=True)
model.load_state_dict(baseline_state,strict=True)
del baseline_state
BASELINE_SHA=sha_file(BASELINE_CHECKPOINT)
print("V33 initialization PASS |",BASELINE_CHECKPOINT,"| SHA",BASELINE_SHA)


V33 initialization PASS | /content/drive/MyDrive/PHD/Datathon/track2_phase2_v32_indicparler_beats_demucs/run_v1/demucs_epoch_06.pt | SHA da6762a5383c838254e82cbadb27e29b6b51872ca29c06018be25ae703b90a7c


## 5 — Event-centred low-LR stabilization; checkpoints resume automatically


In [ ]:
def neg_si_sdr_loss(y,s):
    y=y-y.mean(-1,keepdim=True);s=s-s.mean(-1,keepdim=True)
    target=(y*s).sum(-1,keepdim=True)/(s.square().sum(-1,keepdim=True)+1e-8)*s
    error=y-target
    score=10*torch.log10((target.square().sum(-1)+1e-8)/(error.square().sum(-1)+1e-8))
    return -score.mean()/20.0

def multires_stft_loss(y,s):
    total=0
    for size in (256,512,1024):
        win=torch.hann_window(size,device=y.device)
        Y=torch.stft(y,size,hop_length=size//4,window=win,return_complex=True,pad_mode="constant").abs()
        S=torch.stft(s,size,hop_length=size//4,window=win,return_complex=True,pad_mode="constant").abs()
        total+=(torch.log1p(Y)-torch.log1p(S)).abs().mean()
    return total/3

def enhancement_loss(net,x,s,event_mask):
    y=net(x[:,None])[:,0]
    l_si=neg_si_sdr_loss(y,s)
    l_stft=multires_stft_loss(y,s)
    scale=s.square().mean(-1).sqrt().clamp_min(.01)[:,None]
    l_wave=((y-s)/scale).abs().mean()
    outside=1-event_mask
    preserve=(((y-s).abs()/scale)*outside).sum()/outside.sum().clamp_min(1)
    loss=l_si+.15*l_stft+.10*l_wave+.15*preserve
    return loss,dict(si_sdr=l_si,stft=l_stft,wave=l_wave,preserve=preserve),y

class PairDataset(Dataset):
    def __init__(self,records,training=True):
        self.records=list(records);self.training=training;self.epoch=0
    def set_epoch(self,epoch):self.epoch=int(epoch)
    def __len__(self):return len(self.records)
    def __getitem__(self,i):
        r=self.records[i];x=read_wave_strict(r["noisy_path"]);s=read_wave_strict(r["clean_path"])
        if len(x)!=len(s):raise ValueError("Pair length mismatch")
        evs=event_dicts(r.get("events",[]));n=min(len(x),SEGMENT)
        rng=np.random.default_rng(SEED+i*1000003+self.epoch*10007)
        u=float(rng.random()) if self.training else 0.0
        identity=bool(self.training and u<IDENTITY_PROBABILITY)
        random_crop=bool(self.training and IDENTITY_PROBABILITY<=u<IDENTITY_PROBABILITY+RANDOM_CROP_PROBABILITY)
        if identity:
            x=s.copy();evs=[]
        if random_crop or not evs:
            start=int(rng.integers(0,len(x)-n+1)) if self.training and len(x)>n else max(0,(len(x)-n)//2)
        else:
            e=evs[int(rng.integers(0,len(evs)))] if self.training else evs[0]
            a=max(0,min(len(x)-1,int(e["start"]*SR)));b=max(a+1,min(len(x),int(e["end"]*SR)))
            anchor=int(rng.integers(a,b));offset=int(rng.integers(0,n+1)) if self.training else n//2
            start=max(0,min(len(x)-n,anchor-offset))
        xx=np.pad(x[start:start+n],(0,SEGMENT-n));ss=np.pad(s[start:start+n],(0,SEGMENT-n))
        mask=np.zeros(SEGMENT,np.float32)
        for e in evs:
            a=max(0,int(e["start"]*SR)-start);b=min(n,int(e["end"]*SR)-start)
            if b>a:mask[a:b]=1
        return torch.from_numpy(xx.astype(np.float32)),torch.from_numpy(ss.astype(np.float32)),torch.from_numpy(mask)

pseudo_records=PSEUDO[PSEUDO.split=="pseudo_train"].to_dict("records")
real_records=[]
for r in TRAIN_ROWS.to_dict("records"):
    real_records.append(dict(pair_id="real_"+r["clip_id"],noisy_path=r["noisy_path"],
        clean_path=r["clean_path"],events=r["events"]))
# Roughly one quarter of updates remain anchored in genuine validation pairs.
train_records=pseudo_records+real_records*3
val_records=PSEUDO[PSEUDO.split=="pseudo_val"].to_dict("records")
train_ds=PairDataset(train_records,True);val_ds=PairDataset(val_records,False)
train_dl=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=2,pin_memory=True,drop_last=True)
val_dl=DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=2,pin_memory=True)
print("Training records:",len(train_ds),"| pseudo validation:",len(val_ds))
print("Crop policy: event-centred",EVENT_CENTRED_PROBABILITY,"| random",RANDOM_CROP_PROBABILITY,
      "| clean identity",IDENTITY_PROBABILITY)


Training records: 5964 | pseudo validation: 509
Crop policy: event-centred 0.7 | random 0.15 | clean identity 0.15


In [ ]:
history_path=RUN/"demucs_training_history.csv"
history=pd.read_csv(history_path).to_dict("records") if history_path.exists() else []
existing=sorted(RUN.glob("demucs_v33_epoch_*.pt"))
start_epoch=0
if existing:
    latest=existing[-1];model.load_state_dict(torch.load(latest,map_location="cpu",weights_only=True),strict=True)
    start_epoch=int(latest.stem.rsplit("_",1)[-1]);print("Resuming V33 after epoch",start_epoch)

def flatten_rnns(net):
    for module in net.modules():
        if isinstance(module,nn.RNNBase):module.flatten_parameters()

@torch.inference_mode()
def validation_loss():
    model.eval();flatten_rnns(model);vals=[]
    for x,s,m in val_dl:
        x,s,m=x.to(DEVICE),s.to(DEVICE),m.to(DEVICE)
        with torch.amp.autocast("cuda",enabled=True):vals.append(float(enhancement_loss(model,x,s,m)[0]))
    return float(np.mean(vals))

CANDIDATES={"v32_epoch06":BASELINE_CHECKPOINT}
model.to(DEVICE)
for epoch in range(start_epoch+1,FINETUNE_EPOCHS+1):
    train_ds.set_epoch(epoch)
    optimizer=torch.optim.AdamW(model.parameters(),lr=LR_FULL,weight_decay=1e-5)
    model.train();flatten_rnns(model);values=[];parts_sum=[]
    for x,s,m in tqdm(train_dl,desc=f"V33 multi-event fine-tune {epoch}/{FINETUNE_EPOCHS}"):
        x,s,m=x.to(DEVICE,non_blocking=True),s.to(DEVICE,non_blocking=True),m.to(DEVICE,non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda",enabled=True):loss,parts,_=enhancement_loss(model,x,s,m)
        if not torch.isfinite(loss):raise RuntimeError("Non-finite training loss")
        loss.backward();nn.utils.clip_grad_norm_(model.parameters(),5.,error_if_nonfinite=True);optimizer.step()
        values.append(float(loss.detach()));parts_sum.append([float(parts[k].detach()) for k in ("si_sdr","stft","wave","preserve")])
    val=validation_loss();means=np.mean(parts_sum,axis=0)
    rec=dict(epoch=epoch,stage="full_low_lr",lr=LR_FULL,train_loss=float(np.mean(values)),val_loss=val,
        si_sdr=float(means[0]),stft=float(means[1]),wave=float(means[2]),preserve=float(means[3]))
    history=[r for r in history if int(r["epoch"])!=epoch]+[rec];history=sorted(history,key=lambda r:int(r["epoch"]))
    print(rec)
    path=RUN/f"demucs_v33_epoch_{epoch:02d}.pt";tmp=path.with_suffix(".tmp")
    torch.save({k:v.detach().cpu() for k,v in model.state_dict().items()},tmp);os.replace(tmp,path)
    pd.DataFrame(history).to_csv(history_path,index=False);del optimizer
for p in sorted(RUN.glob("demucs_v33_epoch_*.pt")):CANDIDATES[p.stem]=p
model.cpu();gc.collect();torch.cuda.empty_cache()
print("Candidate checkpoints:",list(CANDIDATES))


V33 multi-event fine-tune 1/3:   0%|          | 0/1491 [00:00<?, ?it/s]

{'epoch': 1, 'stage': 'full_low_lr', 'lr': 2e-06, 'train_loss': -1.286320145461161, 'val_loss': -0.7599402938503772, 'si_sdr': -1.29667398403508, 'stft': 0.011808109469243308, 'wave': 0.07104064652660023, 'preserve': 0.00985705213106564}


V33 multi-event fine-tune 2/3:   0%|          | 0/1491 [00:00<?, ?it/s]

{'epoch': 2, 'stage': 'full_low_lr', 'lr': 2e-06, 'train_loss': -1.3037317317356447, 'val_loss': -0.7641860423609614, 'si_sdr': -1.3133067282292765, 'stft': 0.011409347019524337, 'wave': 0.06812324417826215, 'preserve': 0.007008475496428679}


V33 multi-event fine-tune 3/3:   0%|          | 0/1491 [00:00<?, ?it/s]

{'epoch': 3, 'stage': 'full_low_lr', 'lr': 2e-06, 'train_loss': -1.2950119437246335, 'val_loss': -0.7655051294714212, 'si_sdr': -1.3041889695294506, 'stft': 0.011397300936954095, 'wave': 0.0669097800317953, 'preserve': 0.00517634339168378}
Candidate checkpoints: ['v32_epoch06', 'demucs_v33_epoch_01', 'demucs_v33_epoch_02', 'demucs_v33_epoch_03']


## 6 — Always-on inference and frozen SraVaani evaluation


In [ ]:
LAST_MODEL_CALLS=0
@torch.inference_mode()
def enhance_audio(y):
    global LAST_MODEL_CALLS
    y=np.asarray(y,np.float32);out=np.zeros(len(y),np.float64);den=np.zeros(len(y),np.float64)
    weight=np.maximum(np.hanning(CHUNK_SAMPLES),1e-3);LAST_MODEL_CALLS=0
    for start in range(0,len(y),CHUNK_HOP):
        count=min(CHUNK_SAMPLES,len(y)-start);seg=np.pad(y[start:start+count],(0,CHUNK_SAMPLES-count))
        xt=torch.from_numpy(seg)[None].to(DEVICE)
        z=model(xt[:,None])[:,0][0].float().cpu().numpy()
        if not np.isfinite(z).all():raise RuntimeError("Invalid enhancement; no fallback")
        out[start:start+count]+=z[:count]*weight[:count];den[start:start+count]+=weight[:count];LAST_MODEL_CALLS+=1
        if start+CHUNK_SAMPLES>=len(y):break
    if (den<=0).any():raise RuntimeError("Uncovered samples")
    return (out/den).astype(np.float32)

def activate(path,device=DEVICE):
    model.load_state_dict(torch.load(path,map_location="cpu",weights_only=True),strict=True)
    model.to(device).eval();flatten_rnns(model)


In [ ]:
from transformers import AutoModel
pin=json.loads(ASR_PIN.read_text()); ASR_REV=str(pin['revision'])
if pin.get('repo')!=ASR_REPO or not re.fullmatch('[0-9a-fA-F]{40}',ASR_REV):
    raise ValueError('Invalid existing SraVaani revision pin')
asr_identity=dict(repo=ASR_REPO,revision=ASR_REV,task='transcribe_return_hypotheses',
    pipeline='v29_content_filename_nograd_v1',torch=str(torch.__version__),
    transformers=importlib.metadata.version('transformers'))
ASR_SIGNATURE=sha_json(asr_identity)
token=os.environ.get('HF_TOKEN')
if not token:
    try:
        from google.colab import userdata
        token=userdata.get('HF_TOKEN')
    except Exception:pass
asr_model=AutoModel.from_pretrained(ASR_REPO,revision=ASR_REV,code_revision=ASR_REV,
    trust_remote_code=True,token=token).to(DEVICE).eval()
put_json(RUN/'asr_identity.json',asr_identity)

class FreshASR:
    def __init__(self,model):
        self.model=model;self.cache=ROOT/'track2_phase2_v28_pretrained_wer'/'asr_cache_v28'/ASR_SIGNATURE
        self.cache.mkdir(parents=True,exist_ok=True)
        self.stage=WORK/'asr_inputs';self.stage.mkdir(exist_ok=True)
    def transcribe(self,paths,force=False):
        result={}
        for cid,p in tqdm(paths.items(),desc='Content-audited SraVaani'):
            p=Path(p);h=sha_file(p);dest=self.cache/(h+'.json')
            rec=None
            if dest.exists() and not force:
                trial=json.loads(dest.read_text())
                if trial.get('wav_sha256')==h and trial.get('model_signature')==ASR_SIGNATURE:
                    rec=trial
            if rec is None:
                staged=self.stage/(h+'.wav')
                if not staged.exists() or sha_file(staged)!=h:shutil.copyfile(p,staged)
                if sha_file(staged)!=h:raise IOError('ASR input hash mismatch')
                # One audio per call makes result-to-file association unambiguous.
                with torch.inference_mode(False), torch.no_grad():
                    hyp=self.model.transcribe([str(staged)],return_hypotheses=True)
                text=hyp_text(hyp)
                rec=dict(text=text,wav_sha256=h,model_signature=ASR_SIGNATURE,
                         staged_name=staged.name,source_path=str(p))
                put_json(dest,rec)
            if not isinstance(rec.get('text'),str):raise ValueError('Invalid cached transcript')
            result[cid]=rec
        return result
ASR=FreshASR(asr_model)

def text_tokens(text):
    text=re.sub(r'<[^>]*>|\[[^\]]*\]',' ',text)
    text=unicodedata.normalize('NFC',text).lower()
    return ''.join(' ' if unicodedata.category(c).startswith('P') else c for c in text).split()

def word_counts(ref,hyp):
    # Deterministic minimum-edit alignment; ties prefer substitution, then deletion.
    a,b=text_tokens(ref),text_tokens(hyp);n,m=len(a),len(b)
    dp=np.zeros((n+1,m+1),dtype=np.int32)
    dp[:,0]=np.arange(n+1);dp[0,:]=np.arange(m+1)
    for i in range(1,n+1):
        for j in range(1,m+1):
            dp[i,j]=min(dp[i-1,j-1]+(a[i-1]!=b[j-1]),dp[i-1,j]+1,dp[i,j-1]+1)
    i,j=n,m;sub=delete=insert=hit=0
    while i or j:
        if i and j and a[i-1]==b[j-1] and dp[i,j]==dp[i-1,j-1]:hit+=1;i-=1;j-=1
        elif i and j and dp[i,j]==dp[i-1,j-1]+1:sub+=1;i-=1;j-=1
        elif i and dp[i,j]==dp[i-1,j]+1:delete+=1;i-=1
        else:insert+=1;j-=1
    return dict(words=n,substitutions=sub,deletions=delete,insertions=insert,
                errors=sub+delete+insert,hits=hit)
assert word_counts('a b','a c')['substitutions']==1
assert word_counts('a b','a')['deletions']==1
assert word_counts('a','a b')['insertions']==1

BASELINES={}
def get_reference_decodes(label,rows):
    if label in BASELINES:return BASELINES[label]
    raw={r.clip_id:Path(r.noisy_path) for r in rows.itertuples()}
    raw_h=ASR.transcribe(raw)
    clean_h=ASR.transcribe({r.clip_id:Path(r.clean_path) for r in rows.itertuples()}) if 'clean_path' in rows else None
    BASELINES[label]=(raw_h,clean_h)
    return raw_h,clean_h
get_reference_decodes('synthetic_dev',SYN_DEV)
get_reference_decodes('natural_dev',NAT_DEV)
# Independent uncached re-decode of one noisy/clean pair.
first=SYN_DEV.iloc[0]
check=ASR.transcribe({'noisy':Path(first.noisy_path),'clean':Path(first.clean_path)},force=True)
put_json(RUN/'uncached_pair_check.json',check)
print('First pair fresh decode:',{k:v['text'] for k,v in check.items()})


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/ARTPARK-IISc/SraVaani-1.0.
401 Client Error. (Request ID: Root=1-6acdbfe8-1c51b39234a4de2517173a17;39aa32b5-d2c5-4a04-bd22-2f9f392717b7)

Cannot access gated repo for url https://huggingface.co/ARTPARK-IISc/SraVaani-1.0/resolve/f5dd5358325a5208775b91dad98918e079ea2b27/config.json.
Access to model ARTPARK-IISc/SraVaani-1.0 is restricted. You must have access to it and be authenticated to access it. Please log in.

In [ ]:
def process_rows(rows,directory,checkpoint_sha):
    directory.mkdir(parents=True,exist_ok=True);paths={};audit=[]
    for r in tqdm(rows.to_dict('records'),desc='Always-on enhancement'):
        cid=r['clip_id'];x=read_wave_strict(r['noisy_path'])
        p=directory/(cid+'.wav');rec_path=directory/(cid+'.audit.json')
        key=sha_json(dict(input=hashlib.sha256(x.tobytes()).hexdigest(),checkpoint=checkpoint_sha,impl=IMPLEMENTATION))
        rec=None
        if p.exists() and rec_path.exists():
            trial=json.loads(rec_path.read_text())
            if trial.get('key')==key and trial.get('wav_sha256')==wav_audit(p,len(x)):rec=trial
        if rec is None:
            z=enhance_audio(x);write_pcm(p,z)
            rec=dict(clip_id=cid,key=key,model_calls=LAST_MODEL_CALLS,wav_sha256=wav_audit(p,len(x)),
                unchanged_pcm=bool(np.array_equal(canonical_pcm(x),canonical_pcm(z))),
                clipped_samples=int(np.count_nonzero((z<-1)|(z>32767/32768))),
                change_rms=float(np.sqrt(np.mean((pcm_roundtrip(z)-pcm_roundtrip(x))**2))))
            put_json(rec_path,rec)
        paths[cid]=p;audit.append(rec)
    return paths,audit

def evaluate(label,rows,paths,candidate):
    raw,clean=get_reference_decodes(label,rows)
    enh=ASR.transcribe(paths);details=[];skipped=0
    for r in rows.to_dict('records'):
        cid=r['clip_id'];ref=r['transcript']
        if not text_tokens(ref):skipped+=1;continue
        record=dict(clip_id=cid,reference=ref,raw_text=raw[cid]['text'],enhanced_text=enh[cid]['text'],
            raw_sha=raw[cid]['wav_sha256'],enhanced_sha=enh[cid]['wav_sha256'],
            same_transcript=text_tokens(raw[cid]['text'])==text_tokens(enh[cid]['text']))
        for role,h in [('raw',raw),('enhanced',enh),('clean',clean)]:
            if h is not None:
                record.update({role+'_'+k:v for k,v in word_counts(ref,h[cid]['text']).items()})
                if role=='clean':record['clean_text']=h[cid]['text'];record['clean_sha']=h[cid]['wav_sha256']
        if 'clean_path' in r:
            s=read_wave_strict(r['clean_path']);x=read_wave_strict(r['noisy_path']);z=read_wave_strict(paths[cid])
            record['noisy_si_sdr']=si_sdr(x,s);record['enhanced_si_sdr']=si_sdr(z,s)
        record['error_increase']=record['enhanced_errors']-record['raw_errors']
        details.append(record)
    if not details:raise ValueError('No usable reference transcripts')
    frame=pd.DataFrame(details).sort_values('error_increase',ascending=False)
    frame.to_csv(RUN/(candidate+'_'+label+'_details.csv'),index=False)
    words=int(frame.raw_words.sum());summary=dict(candidate=candidate,subset=label,clips=len(frame),skipped=skipped,reference_words=words,raw_errors=int(frame.raw_errors.sum()),enhanced_errors=int(frame.enhanced_errors.sum()),
        raw_wer_percent=100*frame.raw_errors.sum()/words,
        enhanced_wer_percent=100*frame.enhanced_errors.sum()/words,
        identical_transcript_fraction=float(frame.same_transcript.mean()))
    summary['delta_wer_points']=summary['raw_wer_percent']-summary['enhanced_wer_percent']
    for role in ['raw','enhanced']:
        for typ in ['substitutions','deletions','insertions']:summary[role+'_'+typ]=int(frame[role+'_'+typ].sum())
    if clean is not None:
        summary['clean_wer_percent']=100*frame.clean_errors.sum()/words
        summary['enhanced_si_sdr']=float(frame.enhanced_si_sdr.mean())
        summary['noisy_si_sdr']=float(frame.noisy_si_sdr.mean())
    print(summary);return summary



In [ ]:
summaries=[];selection=[]
for candidate,checkpoint in CANDIDATES.items():
    asr_model.to("cpu");torch.cuda.empty_cache();activate(checkpoint);checkpoint_sha=sha_file(checkpoint)
    syn_paths,syn_audit=process_rows(SYN_DEV,WORK/candidate/"synthetic_dev",checkpoint_sha)
    nat_paths,nat_audit=process_rows(NAT_DEV,WORK/candidate/"natural_dev",checkpoint_sha)
    model.to("cpu");torch.cuda.empty_cache();asr_model.to(DEVICE).eval()
    a=evaluate("synthetic_dev",SYN_DEV,syn_paths,candidate);b=evaluate("natural_dev",NAT_DEV,nat_paths,candidate)
    summaries.extend([a,b]);words=a["reference_words"]+b["reference_words"]
    dwer=100*((a["raw_errors"]+b["raw_errors"])-(a["enhanced_errors"]+b["enhanced_errors"]))/words
    proxy=a["enhanced_si_sdr"]+dwer
    unchanged=any(r["unchanged_pcm"] for r in syn_audit+nat_audit)
    selection.append(dict(candidate=candidate,synthetic_si_sdr=a["enhanced_si_sdr"],pooled_delta_wer_points=dwer,
                          combined_proxy=proxy,unchanged=unchanged))
    pd.DataFrame(syn_audit+nat_audit).to_csv(RUN/(candidate+"_processing_audit.csv"),index=False)
    put_json(RUN/"development_metrics.json",summaries)
eligible_sel=[r for r in selection if not r["unchanged"]]
if not eligible_sel:raise RuntimeError("No genuinely processing checkpoint")
BEST=max(eligible_sel,key=lambda r:r["combined_proxy"])["candidate"]
SELECTED=CANDIDATES[BEST];SELECTED_SHA=sha_file(SELECTED)
put_json(RUN/"selected_model.json",dict(candidate=BEST,checkpoint=str(SELECTED),sha256=SELECTED_SHA,scores=selection,
         criterion="synthetic SI-SDR plus pooled WER improvement points"))
display(pd.DataFrame(selection).sort_values("combined_proxy",ascending=False));print("Selected:",BEST)

# Monitoring is reported once and never used to reselect among candidates.
asr_model.to("cpu");torch.cuda.empty_cache();activate(SELECTED)
sp,sa=process_rows(SYN_MON,WORK/BEST/"synthetic_monitor",SELECTED_SHA)
npth,na=process_rows(NAT_MON,WORK/BEST/"natural_monitor",SELECTED_SHA)
model.to("cpu");torch.cuda.empty_cache();asr_model.to(DEVICE).eval()
monitor=[evaluate("synthetic_monitor",SYN_MON,sp,BEST),evaluate("natural_monitor",NAT_MON,npth,BEST)]
put_json(RUN/"monitor_metrics.json",monitor)


## 7 — Generate the Phase-2 submission ZIP from the selected checkpoint


In [ ]:
if RUN_PHASE2:
    if not CONFIRM_RULES_ALLOW_SOURCE_DERIVED_TRAINING:
        raise RuntimeError("Rules confirmation is required before submission generation")
    ids=sorted(TRANSCRIPT_MAP)
    if len(ids)!=EXPECTED_PHASE2_CLIPS:raise ValueError("Unexpected Phase-2 clip count")
    source=StrictWaveSource(SOURCE_AUDIO,ids)
    asr_model.to("cpu");torch.cuda.empty_cache();activate(SELECTED)
    wavs=WORK/"phase2"/SELECTED_SHA[:12];wavs.mkdir(parents=True,exist_ok=True)
    paths={};frames={};audits=[]
    for cid in tqdm(ids,desc="All-clip always-on V33 enhancement"):
        x=source.load(cid);z=enhance_audio(x);p=wavs/(cid+".wav");write_pcm(p,z)
        audits.append(dict(clip_id=cid,input_sha256=hashlib.sha256(x.tobytes()).hexdigest(),
            wav_sha256=wav_audit(p,len(x)),checkpoint_sha256=SELECTED_SHA,model_calls=LAST_MODEL_CALLS,
            unchanged_pcm=bool(np.array_equal(canonical_pcm(x),canonical_pcm(z))),
            clipped_samples=int(np.count_nonzero((z<-1)|(z>32767/32768)))))
        paths[cid]=p;frames[cid]=len(x)
    source.close();pd.DataFrame(audits).to_csv(RUN/"phase2_processing_audit.csv",index=False)
    if any(r["unchanged_pcm"] for r in audits):raise RuntimeError("Unchanged output found; no bypass permitted")
    model.to("cpu");torch.cuda.empty_cache();asr_model.to(DEVICE).eval();transcripts=ASR.transcribe(paths)
    ZIP=RUN/("submission_Track2_PHASE2_V33_MultiEvent_Demucs_"+BEST+"_"+SELECTED_SHA[:10]+".zip")
    package=pack_submission(paths,transcripts,frames,ZIP)
    put_json(RUN/"submission_manifest.json",dict(version="v33_multievent_eventcentred_demucs",candidate=BEST,
        checkpoint_sha256=SELECTED_SHA,baseline_checkpoint_sha256=BASELINE_SHA,all_clips_enhanced=True,
        routing=False,dictionary_inference=False,beats_in_inference=False,tts_in_inference=False,
        pseudo_pairs=PSEUDO_PAIRS,event_count_distribution=PSEUDO.event_count.value_counts().sort_index().to_dict(),
        source_derived_training_permission_confirmed=True,package=package))
    print("ZIP ready for manual upload:",ZIP)
else:
    print("RUN_PHASE2=False: metrics generated; no submission ZIP created.")


## Outputs

All V33 artifacts are written under:

`/content/drive/MyDrive/PHD/Datathon/track2_phase2_v33_multievent_demucs/run_v1`

The V32 TTS audio, residual profiles, matches, epoch-6 checkpoint and submission
remain untouched. The selected model is allowed to remain `v32_epoch06` if no
V33 checkpoint improves the development combined proxy.
